# AnOxFuse reviewer revision: CPU analyses

This notebook preserves the published AnOxFuse released-test predictions and runs the reviewer analyses that use cached ECFP4 and frozen peptide-adapted ESM-2 representations. It does not require a GPU. Stricter results are reported as sensitivity analyses and do not replace the released benchmark.


In [ ]:
from pathlib import Path
import json, os, sys, zipfile

def safe_extract(archive, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as handle:
        for member in handle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        handle.extractall(destination)

def complete_bundle(path):
    manifest_path = path / "input_manifest.json"
    if not manifest_path.is_file():
        return False
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        return all((path / relative_path).is_file() for relative_path in manifest["files"])
    except Exception:
        return False

candidates = [
    Path.cwd() / "AnOxFuse_Reviewer_Revision_Bundle",
    Path("/data/AnOxFuse_Reviewer_Revision_Bundle"),
    Path.cwd(),
]
ROOT = next((path.resolve() for path in candidates if complete_bundle(path)), None)
if ROOT is None:
    archives = [Path.cwd() / "AnOxFuse_Reviewer_Revision_Bundle.zip", Path("/data/AnOxFuse_Reviewer_Revision_Bundle.zip")]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise RuntimeError("Upload the complete AnOxFuse_Reviewer_Revision_Bundle folder or ZIP.")
    destination = archive.parent
    safe_extract(archive, destination)
    ROOT = destination / "AnOxFuse_Reviewer_Revision_Bundle"
    if not complete_bundle(ROOT):
        if complete_bundle(destination):
            ROOT = destination
        else:
            matches = [path.parent for path in destination.glob("*/input_manifest.json") if complete_bundle(path.parent)]
            if len(matches) != 1:
                raise RuntimeError("Could not locate a complete extracted revision bundle.")
            ROOT = matches[0]
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Revision bundle:", ROOT)


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements_cpu.txt")])
print("Dependencies installed. If this cell changed NumPy or scikit-learn in an already active kernel, restart the kernel once before continuing.")


In [ ]:
import importlib, platform
required = ["numpy", "pandas", "scipy", "sklearn", "lightgbm", "rdkit", "Bio", "parasail", "matplotlib"]
versions = {}
for module_name in required:
    module = importlib.import_module(module_name)
    versions[module_name] = getattr(module, "__version__", "not reported")
print("Python", platform.python_version())
print(versions)


In [ ]:
import runpy
runpy.run_path(str(ROOT / "run_cpu_revision.py"), run_name="__main__")


In [ ]:
import pandas as pd
display(pd.read_csv(ROOT / "revision_outputs" / "cpu" / "tables" / "published_baseline_metrics.csv").round(4))
display(pd.read_csv(ROOT / "revision_outputs" / "cpu" / "tables" / "identity_threshold_bootstrap_summary.csv").query("metric in ['roc_auc','average_precision','mcc']").round(4))
display(pd.read_csv(ROOT / "revision_outputs" / "cpu" / "tables" / "strict_cluster_disjoint_metrics.csv").round(4))
display(pd.read_csv(ROOT / "revision_outputs" / "cpu" / "tables" / "repeated_grouped_fusion_summary.csv").round(4))
